# `create_agent()`와 Tool Calling

* **`create_agent()`** → Tool Calling 과정을 자동으로 관리하는 Agent 생성
* **`tools=[get_weather]`** → Agent가 사용할 Tool 등록
* **`agent.invoke()`** → LLM → Tool 호출 → 결과 전달 → 최종 답변 과정을 실행

```text
agent.invoke()
   ↓
LLM 판단
   ↓
Tool 호출
   ↓
결과 전달
   ↓
최종 답변
```

In [3]:
from dotenv import load_dotenv

load_dotenv()

from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': "It's always sunny in San Francisco!"}]


### 다른 예제

시스템 프롬프트 정의

In [4]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

툴 정의, 툴을 사용 하면 사용자가 정의한 함수를 호출하여 모델이 외부 시스템과 상호 작용할 수 있음.

In [5]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

모델 구성

In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.5",
    temperature=0.5,
    timeout=300,
    max_tokens=25000,
)

에이전트가 이전 대화와 맥락을 기억할 수 있도록 메모리 추가

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

일반 에이전트와 딥 에이전트 비교를 위해 딥 에이전트도 함께 생성
```txt
              같은 질문
                  ↓
        ┌─────────┴─────────┐
        ↓                   ↓
 create_agent          deep_agent
        ↓                   ↓
      결과 A              결과 B
        └─────────┬─────────┘
                  ↓
              비교해보기
```

수행할 작업 내용:
- Gutenberg에서 《The Great Gatsby》 전체 텍스트를 가져오기
- 전체 텍스트에서 Gatsby가 들어간 줄(line)이 몇 개인지 세기
- 같은 줄에 Gatsby가 3번 있어도 1줄로 계산
- Daisy가 처음 등장하는 줄 번호 찾기
- 책 내용을 2문장으로 요약하기
- 단, 정확한 숫자를 확인할 수 없으면 추측하지 말고 null로 표시

목적:
- Agent에게 단순 질문이 아니라, 외부 자료를 가져오고 → 자료를 분석하고 → 결과를 검증하는 복합 작업을 시켜보는 것

In [8]:
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

print("Running create_agent...", flush=True)
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
print("Running create_deep_agent...", flush=True)
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print("\ncreate_agent:")
print(agent_result["messages"][-1].content_blocks)
print("\ncreate_deep_agent:")
print(deep_agent_result["messages"][-1].content_blocks)

Running create_agent...
Running create_deep_agent...

create_agent:
[{'type': 'text', 'text': '{\n  "gatsby_lines_with_substring": null,\n  "first_line_containing_daisy": 180,\n  "synopsis": "Narrated by Nick Carraway, the novel follows Jay Gatsby, a wealthy Long Island host whose lavish parties conceal his fixation on renewing a past romance with Daisy Buchanan. The story traces the entanglements among Gatsby, Daisy, Tom Buchanan, Jordan Baker, Myrtle Wilson, and George Wilson, ending in tragedy and Nick’s disillusionment with the social world he has observed.",\n  "how_you_computed_counts": "I fetched the Gutenberg plain-text file from the provided URL. I manually counted preserved line breaks from the start of the fetched text to the first line containing `Daisy`, which is the line beginning `Buchanans. Daisy was my second cousin...`, yielding line 180. I cannot verify an exact count for all complete-file lines containing `Gatsby` with the available tool, because I have text-fetch a

### 결과 비교:
- create_agent 결과
        - Gatsby가 포함된 줄의 개수 → 계산하지 못해서 null
        - Daisy가 처음 등장하는 줄 → 180번째 줄
        - 소설 요약 → 개츠비와 데이지의 관계, 톰·조던·머틀·조지 등이 얽히면서 비극으로 끝나는 이야기라고 요약
        - 계산 과정 → Gutenberg 텍스트를 가져왔지만 전체 파일에서 Gatsby가 들어간 줄을 정확하게 세는 도구가 없어서 포기
        - 즉, 할 수 있는 것만 하고 어려운 것은 null로 반환
- create_deep_agent 결과
        - Gatsby가 포함된 줄 → 258줄
        - Daisy가 처음 등장하는 줄 → 181번째 줄
        - 소설 요약 → 닉의 시점에서 개츠비와 데이지, 톰을 중심으로 계급·부·환상과 비극을 다룬다고 요약
        - 계산 과정 → 파일 전체를 저장한 뒤 grep으로 Gatsby를 직접 검색해서 258개를 계산
        - Daisy도 실제 파일의 앞부분을 확인해서 181번째 줄이 맞는지 검증

### 핵심 차이:
일반 create_agent는 "도구를 사용해서 답변"하는 수준이고, create_deep_agent는 필요한 작업을 스스로 쪼개서 파일 저장 → 검색 → 개수 확인 → 검증까지 수행함.